### CLASSIFICATION DES IMAGES PAR CLASSES ET SOUS-CLASSES

In [ ]:
#Importration des librairies . 
from sklearn import datasets 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler as sc
from sklearn.model_selection import train_test_split
import random
from sklearn.neural_network import MLPClassifier as MLPC
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn import svm
import keras
from keras import layers
from keras import ops
from keras.optimizers import SGD
import glob
from PIL import Image
from sklearn.preprocessing import LabelBinarizer



#### 1ère partie : Classification un modèle de machine learning sur les caractéristiques extraites.

In [ ]:
#Imortation du dataset. 

Xdf = pd.read_csv("database.csv") 

#On visualise la structure du dataset
n,p = Xdf.shape
print(f"Taille du dataset : ({n}, {p})\n")
Xdf.head()


On remarque que notre dataset est constitué de 2688 lignes et de 30 colonnes, la première représente le numéro de l'image les 2 suivantes sont respectivement les classes et sous-classes de chaque image et
les 27 suivantes (de la 4 à la 30e) représentent les caractéristiques



In [ ]:
Xdf.dtypes


In [ ]:
Xdf.info()

In [ ]:
Xdf.describe()

Chaque feature est du type float ( variable quantitaitve) la classe et la sous-classe sont des object (texte non encodé) tandis que le numéro est un entier entre 0 et 2687 et le dataset ne présente pas de valeur manquante

In [ ]:
# Séparation des features du dataset et des colonnes cibles
Xdf["Group1"] = Xdf["SousClasse"].astype("category")
Xdf["Group2"] = Xdf["Classe"].astype("category")

Xdf_numeric = Xdf.iloc[:,1:].select_dtypes(include="number") #En excluant la colonne numéro
y_sous_classe = Xdf["Group1"]
y_classe = Xdf["Group2"]

##### Analyse descriptive

In [ ]:

#Tracer des boxplot 
plt.figure(figsize=(20, 12)) # Ajuste la taille pour 27 features
Xdf_numeric.boxplot()

# 4. Améliorer la lisibilité
plt.title('Boxplot de toutes les 27 caractéristiques')
plt.xticks(rotation=90) # Fait pivoter les noms des features
plt.show()

In [ ]:
#Chaque boxplot seul et par classe et sous-classe
for column in Xdf_numeric:
    #Par sous-classes
    plt.figure(figsize=(14, 4)) 
    sns.boxplot(x="Group1", y=column, data=Xdf) 

    plt.title(f"Distribution de {column} par Sous-Classe")
    plt.show()

    #Par classes
    plt.figure(figsize=(6, 4)) 
    sns.boxplot(x="Group2", y=column, data=Xdf) 

    plt.title(f"Distribution de {column} par Classe")
    plt.show()

In [ ]:
#On calcule les corrélation linéaire par couple de features
plt.figure(figsize=(15, 12))

sns.heatmap(
    Xdf_numeric.corr(),  
    annot=False,         
    cmap='coolwarm',     
    vmin=-1,             
    vmax=1               
)

plt.title('Heatmap de Corrélation des 27 Caractéristiques')
plt.tight_layout() 
plt.show()

On remarque que certaines (les plus proches) des 24 premières caractéristiques sont plutot très positivement corrélées entre elles et pour les 3 dernières seules Cr etCb son négativement corrélées

In [ ]:
#On divise les données en données d'entrainement et de test
X_train_classe , X_test_classe, y_train_classe, y_test_classe = train_test_split(Xdf_numeric,y_classe,test_size=0.2,random_state=42)
X_train_sous_classe , X_test_sous_classe, y_train_sous_classe, y_test_sous_classe = train_test_split(Xdf_numeric,y_sous_classe,test_size=0.2,random_state=42)

In [ ]:
#On va réaliser l'ACP et l'ALD sur chaque jeu de données de test
scaler = sc()
X_test_classe_scaled = scaler.fit_transform(X_test_sous_classe)
X_test_sous_classe_scaled = scaler.fit_transform(X_test_classe)

acp = PCA()

X_acp = acp.fit_transform(X_test_sous_classe_scaled,y_test_sous_classe)


In [ ]:
plt.figure() 
# Graphique du pouvoir de la variance expliquee
var_cum = [0]
for var in acp.explained_variance_ratio_:
    var_cum.append(var_cum[-1]+var)
var_cum.pop(0)
nb_axe_dis = len(X_acp[0])
print(nb_axe_dis)
plt.bar(np.arange(1, nb_axe_dis+1), acp.explained_variance_ratio_) 
plt.plot(np.arange(1,nb_axe_dis+1),var_cum)
plt.ylabel("% de variance expliquée") 
plt.xlabel("Nombre de variables") 
plt.show() 


In [ ]:
#projection sur les deux premiers axes et tracé du nuage de points

plt.figure(figsize=(16,10))
y_series = pd.Series(y_test_sous_classe).astype('category')
class_names = y_series.cat.categories
num_classes = len(class_names)

# Générer un jeu de couleurs distinctes
colors = plt.cm.get_cmap('tab10', num_classes)

# --- 2. Calcul des Centres de Gravité (Centroids) ---
# Mettez vos données 27D (avant ACP) et vos étiquettes dans un DataFrame
df_27d = pd.DataFrame(X_test_sous_classe_scaled) # REQUIERT X_train_scaled
print("ici",len(X_test_sous_classe_scaled))
df_27d['classe'] = y_series.values

# Calculer la moyenne (centre de gravité) en 27 dimensions
centroids_27d = df_27d.groupby('classe').mean().values

# Projeter ces centres 27D sur vos 2 axes ACP
# REQUIERT votre objet 'pca'
centroids_2d = acp.transform(centroids_27d) 

# --- 3. Création du Graphique ---
plt.figure(figsize=(16, 10))

# Boucle pour tracer chaque classe séparément (la clé pour la légende)
for i, class_name in enumerate(class_names):
    # Créer un masque pour ne sélectionner que les points de cette classe
    mask = (y_series == class_name)
    
    # Tracer les points de cette classe
    plt.scatter(
        X_acp[mask, 0],    # Coordonnées X des points de la classe
        X_acp[mask, 1],    # Coordonnées Y des points de la classe
        color=colors(i),   # Couleur spécifique pour cette classe
        label=class_name,  # Nom de la classe pour la légende
        alpha=0.6          # Transparence pour voir les superpositions
    )

# --- 4. Tracer les Centres de Gravité ---
plt.scatter(
    centroids_2d[:, 0], # Coordonnées X des centres
    centroids_2d[:, 1], # Coordonnées Y des centres
    marker='X',         # Marqueur en forme de 'X'
    c='black',          # Couleur noire
    s=250,              # Grande taille
    label='Centres de Gravité', # Étiquette pour la légende
    zorder=10           # 'zorder' pour s'assurer qu'ils sont au-dessus
)

# --- 5. Titre, Étiquettes et Légende ---
plt.title('Projection ACP des Sous-Classes (2 axes) avec Centres de Gravité', fontsize=18)
plt.xlabel('Axe principal 1') 
plt.ylabel('Axe principal 2') 
plt.legend(loc='best', markerscale=1.0) # Ajoute la légende
plt.grid(True, linestyle='--', alpha=0.5) # Ajoute une grille
plt.axhline(0, color='grey', linewidth=0.5) # Ligne à Y=0
plt.axvline(0, color='grey', linewidth=0.5) # Ligne à X=0

plt.show()

In [ ]:
#On va réaliser une ALD par sous-classe
ald = LinearDiscriminantAnalysis()
X_ald = ald.fit_transform(X_test_sous_classe_scaled,y_test_sous_classe)

#Pouvoir dsicirminant
plt.figure() 
# Graphique du pouvoir discriminant de l'axe
nb_axe_dis = len(pd.Series(y_test_sous_classe).cat.categories)-1
print(nb_axe_dis)
var_cum = [0]
for var in ald.explained_variance_ratio_:
    var_cum.append(var_cum[-1]+var)
var_cum.pop(0)
nb_axe_dis = len(X_ald[0])
plt.bar(np.arange(1, nb_axe_dis+1), ald.explained_variance_ratio_) 
plt.plot(np.arange(1, nb_axe_dis+1), var_cum) 
plt.ylabel("pouvoir discriminant") 
plt.xlabel("Nombre de facteurs") 
plt.show()

#Trace des individu
plt.figure(figsize=(16,10))
y_series = pd.Series(y_test_sous_classe).astype('category')
class_names = y_series.cat.categories
num_classes = len(class_names)

# Générer un jeu de couleurs distinctes
colors = plt.cm.get_cmap('tab10', num_classes)

# --- 2. Calcul des Centres de Gravité (Centroids) ---
# Mettez vos données 27D (avant ACP) et vos étiquettes dans un DataFrame
df_27d = pd.DataFrame(X_test_sous_classe_scaled) # REQUIERT X_train_scaled
df_27d['classe'] = y_series.values

# Calculer la moyenne (centre de gravité) en 27 dimensions
centroids_27d = df_27d.groupby('classe').mean().values

# Projeter ces centres 27D sur vos 2 axes ACP
# REQUIERT votre objet 'pca'
centroids_2d = acp.transform(centroids_27d) 

# --- 3. Création du Graphique ---
plt.figure(figsize=(16, 10))

# Boucle pour tracer chaque classe séparément (la clé pour la légende)
for i, class_name in enumerate(class_names):
    # Créer un masque pour ne sélectionner que les points de cette classe
    mask = (y_series == class_name)
    
    # Tracer les points de cette classe
    plt.scatter(
        X_ald[mask, 0],    # Coordonnées X des points de la classe
        X_ald[mask, 1],    # Coordonnées Y des points de la classe
        color=colors(i),   # Couleur spécifique pour cette classe
        label=class_name,  # Nom de la classe pour la légende
        alpha=0.6          # Transparence pour voir les superpositions
    )

# --- 4. Tracer les Centres de Gravité ---
plt.scatter(
    centroids_2d[:, 0], # Coordonnées X des centres
    centroids_2d[:, 1], # Coordonnées Y des centres
    marker='X',         # Marqueur en forme de 'X'
    c='black',          # Couleur noire
    s=250,              # Grande taille
    label='Centres de Gravité', # Étiquette pour la légende
    zorder=10           # 'zorder' pour s'assurer qu'ils sont au-dessus
)

# --- 5. Titre, Étiquettes et Légende ---
plt.title('Projection ACP des Sous-Classes (2 axes) avec Centres de Gravité', fontsize=18)
plt.xlabel('Axe Discriminant 1') 
plt.ylabel('Axe Discriminant 2') 
plt.legend(loc='best', markerscale=1.0) # Ajoute la légende
plt.grid(True, linestyle='--', alpha=0.5) # Ajoute une grille
plt.axhline(0, color='grey', linewidth=0.5) # Ligne à Y=0
plt.axvline(0, color='grey', linewidth=0.5) # Ligne à X=0

plt.show()



In [ ]:
#On va réaliser une ALD par Classes
ald = LinearDiscriminantAnalysis()
X_ald = ald.fit_transform(X_test_sous_classe_scaled,y_test_classe)

#Pouvoir dsicirminant
plt.figure() 
# Graphique du pouvoir discriminant de l'axe
nb_axe_dis = len(pd.Series(y_train_classe).cat.categories)-1
print(nb_axe_dis)
var_cum = [0]
for var in ald.explained_variance_ratio_:
    var_cum.append(var_cum[-1]+var)
var_cum.pop(0)
nb_axe_dis = len(X_ald[0])
plt.bar(np.arange(1, nb_axe_dis+1), ald.explained_variance_ratio_) 
plt.plot(np.arange(1, nb_axe_dis+1), var_cum) 
plt.ylabel("pouvoir discriminant") 
plt.xlabel("Nombre de facteurs") 
plt.show()

#Trace des individu
plt.figure(figsize=(16,10))
y_series = pd.Series(y_test_classe).astype('category')
class_names = y_series.cat.categories
num_classes = len(class_names)

# Générer un jeu de couleurs distinctes
colors = plt.cm.get_cmap('tab10', num_classes)

# --- 2. Calcul des Centres de Gravité (Centroids) ---
# Mettez vos données 27D (avant ACP) et vos étiquettes dans un DataFrame
df_27d = pd.DataFrame(X_test_sous_classe_scaled) # REQUIERT X_train_scaled
df_27d['classe'] = y_series.values

# Calculer la moyenne (centre de gravité) en 27 dimensions
centroids_27d = df_27d.groupby('classe').mean().values

# Projeter ces centres 27D sur vos 2 axes ACP
# REQUIERT votre objet 'pca'
centroids_2d = acp.transform(centroids_27d) 

# --- 3. Création du Graphique ---
plt.figure(figsize=(16, 10))

# Boucle pour tracer chaque classe séparément (la clé pour la légende)
for i, class_name in enumerate(class_names):
    # Créer un masque pour ne sélectionner que les points de cette classe
    mask = (y_series == class_name)
    
    # Tracer les points de cette classe
    plt.scatter(
        X_ald[mask, 0],    # Coordonnées X des points de la classe
        np.random.randint(-4,4,len(X_ald[mask, 0])),    # Coordonnées Y des points de la classe
        color=colors(i),   # Couleur spécifique pour cette classe
        label=class_name,  # Nom de la classe pour la légende
        alpha=0.6          # Transparence pour voir les superpositions
    )

# --- 4. Tracer les Centres de Gravité ---
plt.scatter(
    centroids_2d[:, 0], # Coordonnées X des centres
    np.random.randint(-4,4,len(centroids_2d[:, 0])), # Coordonnées Y des centres
    marker='X',         # Marqueur en forme de 'X'
    c='black',          # Couleur noire
    s=250,              # Grande taille
    label='Centres de Gravité', # Étiquette pour la légende
    zorder=10           # 'zorder' pour s'assurer qu'ils sont au-dessus
)

# --- 5. Titre, Étiquettes et Légende ---
plt.title('Projection ALD des Classes (2 axes) avec Centres de Gravité', fontsize=18)
plt.xlabel('Axe Discriminant 1') 
plt.ylabel('Axe Discriminant 2') 
plt.legend(loc='best', markerscale=1.0) # Ajoute la légende
plt.grid(True, linestyle='--', alpha=0.5) # Ajoute une grille
plt.axhline(0, color='grey', linewidth=0.5) # Ligne à Y=0
plt.axvline(0, color='grey', linewidth=0.5) # Ligne à X=0

plt.show()



#### 2e Partie : 

In [ ]:
# Supposons que X_train_sous_classe et y_train_sous_classe existent

# 1. Définir le modèle et la grille de paramètres
mlpc = MLPC(max_iter=1000, random_state=42) # Ajout du random_state pour la reproductibilité
params = {
    "hidden_layer_sizes": [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 150, 300, 500, 1000 ],
    "activation": ('identity', 'logistic', 'tanh', 'relu')
}

# 2. Configurer le GridSearchCV
# Notez le cv=5 et scoring='balanced_accuracy' ICI
gridSearchClassifier = GridSearchCV(
    mlpc,
    params,
    verbose=4,
    return_train_score=True,
    n_jobs=-1,  # Utilise tous les processeurs
    cv=5,       # C'est ICI qu'on définit la validation croisée
    scoring='balanced_accuracy'
)

# 3. Lancer la recherche des meilleurs paramètres
print("Début du GridSearchCV...")
# On entraîne le GridSearchCV directement sur les données d'entraînement
gridSearchClassifier.fit(X_train_sous_classe, y_train_sous_classe)

print("Recherche terminée !")

# 4. Afficher les résultats
print("--- Meilleurs Résultats ---")
print(f"Meilleur score (balanced_accuracy) : {gridSearchClassifier.best_score_:.4f}")
print(f"Meilleurs paramètres : {gridSearchClassifier.best_params_}")

# 5. Récupérer le meilleur modèle
# Le meilleur modèle, déjà ré-entraîné sur TOUT X_train_sous_classe
# avec les meilleurs paramètres, est disponible dans .best_estimator_
meilleur_modele_mlpc = gridSearchClassifier.best_estimator_

# Vous pouvez maintenant utiliser 'meilleur_modele_mlpc' pour faire
# des prédictions sur votre jeu de test (X_test)
# y_pred = meilleur_modele_mlpc.predict(X_test)

In [ ]:
##On va mettre en place un classifieur SVM
# Supposons que X_train_sous_classe et y_train_sous_classe existent

# 1. Définir le modèle et la grille de paramètres
svmc = svm.SVC(max_iter=10000, random_state=42) # Ajout du random_state pour la reproductibilité
params = {
    "C": [1, 5, 10, 50, 100, 1000, 10000],
    "kernel": ('linear', 'poly', 'rbf', 'sigmoid')
}

# 2. Configurer le GridSearchCV
# Notez le cv=5 et scoring='balanced_accuracy' ICI
gridSearchClassifier = GridSearchCV(
    svmc,
    params,
    verbose=4,
    return_train_score=True,
    n_jobs=-1,  # Utilise tous les processeurs
    cv=5,       # C'est ICI qu'on définit la validation croisée
    scoring='balanced_accuracy'
)

# 3. Lancer la recherche des meilleurs paramètres
print("Début du GridSearchCV...")
# On entraîne le GridSearchCV directement sur les données d'entraînement
gridSearchClassifier.fit(X_train_sous_classe, y_train_sous_classe)

print("Recherche terminée !")

# 4. Afficher les résultats
print("--- Meilleurs Résultats ---")
print(f"Meilleur score (balanced_accuracy) : {gridSearchClassifier.best_score_:.4f}")
print(f"Meilleurs paramètres : {gridSearchClassifier.best_params_}")

# 5. Récupérer le meilleur modèle
# Le meilleur modèle, déjà ré-entraîné sur TOUT X_train_sous_classe
# avec les meilleurs paramètres, est disponible dans .best_estimator_
meilleur_modele_mlpc = gridSearchClassifier.best_estimator_

# Vous pouvez maintenant utiliser 'meilleur_modele_mlpc' pour faire
# des prédictions sur votre jeu de test (X_test)
# y_pred = meilleur_modele_mlpc.predict(X_test)


In [ ]:
#On teste sur les données de test
#On entraine le meilleur modèle 
mlpc_best = MLPC(hidden_layer_sizes=150,activation="logistic")
mlpc_best.fit(X_train_sous_classe,y_train_sous_classe)

b_acc = mlpc_best.score(X_test_sous_classe,y_test_sous_classe)
print(b_acc)

svmc_best = svm.SVC(C=1,kernel='linear')
svmc_best.fit(X_test_sous_classe,y_test_sous_classe)

b_acc  = svmc_best.score(X_test_sous_classe,y_test_sous_classe)
print(b_acc)

# DEUXIEME PARTIE : DEEP LEARNING

In [ ]:
#On va charger les images
path = "C:/Users/emman/OneDrive/Desktop/IESE 5/Machine Learning/projet2/images_128/images_128/"

#On sépare en données d'entrainement et données de test
train_images = []
train_labels = y_train_sous_classe
test_images = []
test_labels = y_test_sous_classe
for ligne in X_train_sous_classe.index:
    train_images.append(np.asarray(Image.open(f"{path}{ligne}.jpg")))

for ligne in X_test_sous_classe.index:
    test_images.append(np.asarray(Image.open(f"{path}{ligne}.jpg")))


In [ ]:
# On transforme la liste en tableau NumPy
train_images = np.array(train_images)
test_images = np.array(test_images)

# Bonne pratique : Normaliser les pixels entre 0 et 1 (au lieu de 0-255)
train_images = train_images.astype('float32') / 255.0
test_images = test_images.astype('float32') / 255.0


In [ ]:
encoder = LabelBinarizer()
train_labels = encoder.fit_transform(train_labels)
test_labels = encoder.fit_transform(test_labels)
# On récupère le nombre exact de classes pour le modèle
nb_classes = len(encoder.classes_)

cnn_model = keras.Sequential()

# Couche 1
cnn_model.add(layers.Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_uniform', 
                            padding='same', input_shape=(128, 128, 3)))
cnn_model.add(layers.MaxPooling2D(2, 2))

# Couche 2
cnn_model.add(layers.Conv2D(8, (5, 5), activation='relu', kernel_initializer='he_uniform', 
                            padding='same'))
cnn_model.add(layers.MaxPooling2D(2, 2))

cnn_model.add(layers.Dropout(0.2))
cnn_model.add(layers.Flatten())

cnn_model.add(layers.Dense(128, activation='relu', kernel_initializer='he_uniform'))
cnn_model.add(layers.Dropout(0.2))

cnn_model.add(layers.Dense(nb_classes, activation='softmax')) 

opt = SGD(learning_rate=0.001, momentum=0.9)

cnn_model.compile(optimizer=opt, 
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

In [ ]:
# --- 4. ENTRAÎNEMENT ---
history = cnn_model.fit(train_images, train_labels, epochs=5)

In [ ]:
# -- TEST -- 
test_loss, test_acc = cnn_model.evaluate(test_images, test_labels, verbose=2)
print(f"Test accuracy: {100*test_acc:.2f}%")